## Solving RTE with frequency dependent absorption by ISIF

  Initialize $ J^0_0 $.
	
  Find $T^{m}(z)$ such that $\int_0^\infty\sigma_a B_\nu(T^{m})d\nu =\int_0^\infty\sigma_a J_0^{m}d\nu$.
	
  Compute $c_0^{m} =  -q_0 \mu_s c_s B_\nu(T_s)e^{-\frac{\kappa_\nu Z}{\mu_s}} -q_0 \int_0^Z E_2(\kappa_\nu z)\left( \sigma_a B_\nu(T^m(z)) +  \sigma_s J_0^m(z)\right)d z$.

    
  Compute $ J_0^{m+1}(z)=  \frac{c'_e}2 B_\nu(T_e)E_3(\kappa_\nu z) +\frac{c_0^m}2 E_2( \kappa_\nu z)
	+ \frac{c_s}2_\nu(T_s)\sigma_s e^{-\frac{\kappa_\nu(Z-z)}{\mu_s}}
	+  \frac2\int_0^Z E_1( \kappa_\nu|z'-z|)\left[ \sigma_a B_\nu(T^m(z')) +  \sigma_s J_0(^mz')\right]d z'.$
  
### Definition of the physical and algorithmic constants

Note that Nz=10 is to little. One should set Nz>30 at least, but then the computing time is big:  Better use the vectorial version

In [1]:
import numpy as np
from scipy.integrate import quad
import os
import sys
import time

verbose =0 # Set to =1 for detailed output during execution

# Constants from original C++ code
Ce = 2.0 # Infrared intensity
Cs = 2e-6 # Solar intensity of collimated light
drho = -0.7 # gradient of density with altitude
Zatmo = 1.2 #Top of Atmosphere
Z = Zatmo * (1.0 + drho * Zatmo / 2.0) # computational TAO due to optical thickness
Bscale = 1.4744e-8 # scaling factor for I and J0...
Tscale = 4798.0 #scaling factor for the temperature
pi = np.pi
stefan = pi**4 / 15.0 # Stefan constant

Te = (273.0 + 18.0) / Tscale
Ts = 5798.0 / Tscale
q0 = -0.3 # Albedo intensity
mus = 0.5 # Angle of collimated light
z1 = Z * 0.5 # Cloud base
z2 = Z * 0.7 # Cloud top
z3 = Z * 0.8 # Rayleigh scttering base
nu1 = 0.5 # cloud is opaque between nu1 and nu2
nu2 = 1.0
lambda_val = 0.5  # Cloud opacity parameter

# Grid parameters
Nz = 10 # Number of discretization points in the vertical direction
dz = Z / (Nz - 1)
kmax = 10 # Number of ISIF iterations
newton = 50 # Max number of Newton iterations
epsdycho = 1e-4 # Precision for dichotomy method
epsnewton = 1e-10 # Precision for Newton method
kappamin = 0.001 # Avoid zero opacity

# Initialize arrays
nu = []
kappanu = []
T = np.zeros(Nz)

# File paths
basedir = ""
mykappafile = os.path.join(basedir, "_kappa.txt") # User's choice 
myresulttemperature = os.path.join(basedir, "temperaturec")
myresultmeanintensity = os.path.join(basedir, "imean0")


## Scattering functions

The background scattering is 0.3 but there is a cloud between z1 and z2 and there is a Rayleigh scattering above z3 for frequencies bewteen nu1 and nu2

## Absoption due to a cloud

cloud(z) returns the increase of absorption at z.

In [2]:

def sqr(x):
    return x * x

# Scattering function
def ascat(z,nu):  # background scattering + cloud scattering
    aux= 0.3 + 0.3*(z2-z)*(z-z1)*(z>z1)*(z<z2)*4/sqr(z1-z2) \
        + 0.3*(nu<nu2)*(nu>nu1)*sqr(4*(nu-nu1)*(nu-nu2)/sqr(nu2-nu1))*(z>z3)  #+ Rayleigh
    return aux

# Cloud function
def cloud(z):
    return 1.0 + lambda_val * ((z > z1) - (z > z2))

# Integral of the cloud function
def scloud(z, zp):
    aux= zp - z + lambda_val * ((zp > z1) * (zp - z1) - (zp > z2) * (zp - z2) - (z > z1) * (z - z1) + (z > z2) * (z - z2))
    return np.abs(aux)

## Exponential Functions and Planck functions

In [3]:

# Exponential integrals (optimized for speed)
def expint_E1(t):
    t1 = abs(t)+1e-10
    ak = t1
    soNtaue =- 0.577215664901533 - np.log(t1) + ak
    for k in range(2, 10):
        ak *= -t1 * (k - 1) / (k * k)
        soNtaue += ak
    return soNtaue

def expint_E2(t=1.0):
    return np.exp(-t) - t * expint_E1(t)

def expint_E3(t=1.0):
    return (np.exp(-t) - t * expint_E2(t)) / 2.0

# Blackbody functions
def Bsun(nu):
    return nu**3 / (np.exp(nu / Ts) - 1.0)

def Bearth(nu):
    return nu**3 / (np.exp(np.fmin(nu / Te, 100.0)) - 1.0)

def BB(nu, T):
    return nu**3 / (np.exp(np.fmin(nu / T, 100.0)) - 1.0)

def dBB(nu, T):
    a =np.exp(np.fmin(nu / T, 100.0))
    return a * sqr(nu**2 / np.fmax(1e-30, a - 1.0) / T)

## Read the function $\nu\to \kappa_\nu$

Note that the complete absorption is $\rho(z)\kappa_\nu$ but the change of variable to optical depth makes $\rho$ disappear.  The function checks also the Stefan formula which is not valid when $\kappa$ depends on $\nu$, but it could debunk a gross error.

In [4]:
def readkappa(mykappafile):
    print(f"Reading kappa(nu) from file {mykappafile}")
    with open(mykappafile, 'r') as kappafile:
        j = -1
        for line in kappafile:
            parts = line.split('\t')
            wavel, kappaux = map(float, line.strip().split())
            kappanu.append(max(kappaux, kappamin))
            nu.append(3.0/wavel)
            j += 1
    print(f"Number of frequencies {j}")
    
    auxe = 0.0
    auxs = 0.0
    for k in range(1, j):
        auxe += BB(nu[k], Te) * (nu[k] - nu[k-1])
        auxs += BB(nu[k], Ts) * (nu[k] - nu[k-1])
    
    print(f"Check Stefan id for the sun {auxs} {stefan * (Ts**4)}")
    print(f"Check Stefan id for the earth {auxe} {stefan * (Te**4)}")
    return j

## $J_0$ is computed from the current right hand side. This is the first important function of the ISIF algorithm

In [5]:
def updateJ():
    global J0,J0old
    for jnu in range(jmax):
        S = np.zeros(Nz)
        kappanuj = kappanu[jnu]
        nuj = nu[jnu]
        bearthnu = Ce * Bearth(nu[jnu])
        bsunnu = Cs * Bsun(nu[jnu])
        c0 = -q0 * bsunnu * mus * np.exp(-scloud(0.0, Z) * kappanuj / mus)
        
        for i in range(Nz):
            z = i * dz
            kappanuz = kappanuj * cloud(z)
            sigs = kappanuz * ascat(z, nuj)
            siga = kappanuz - sigs
            S[i] = sigs * J0old[jnu][i] + siga * BB(nuj, T[i])
            c0 -= q0 * expint_E2(kappanuj * scloud(0.0, z)) * S[i] * dz
        
        for i in range(Nz):
            z = i * dz
            J0z1 = bearthnu * expint_E3(kappanuj * scloud(0.0, z)) / 2.0
            J0z2 = bsunnu * np.exp(-kappanuj * scloud(z, Z) / mus) / 2.0
            J0z3 = c0 * expint_E2(kappanuj * scloud(0.0, z)) / 2.0
            J0z4=0
            for j in range(1, Nz):
                zp = j * dz
                Sj4 = (S[j] + S[j-1])/ 4.0
                J0z4 += expint_E1(kappanuj * scloud(zp, z )+ 0.5 * dz) * Sj4*dz
            J0[jnu][i] = J0z1+J0z2+J0z3+J0z4
    
    return 0


## Solution of the temperature equation

One should read gen(T) first.  It calls the dichotomy first and then the Newton algorithm.


In [6]:

# Root function for dychotomy
def root(rhs, T0, i):
    myeq = -rhs
    for j in range(1, jmax):
        myeq += (1.0 - ascat(i * Z / (Nz - 1.0), nu[j])) * kappanu[j] * BB((nu[j] + nu[j-1]) / 2.0, T0) * (nu[j] - nu[j-1])
    return myeq

# RHS for temperature equation
def rhsTeq(i):
    rhs = 0.0
    for j in range(1, jmax):
        rhs += (1.0 - ascat(i * dz, nu[j])) * kappanu[j] * J0[j][i] * (nu[j] - nu[j-1])
    return rhs

# Get temperature via dichotomy
def getTbydycho(rhs, i, Tstart):
    Taux, T0, T1 = Tstart, Tstart, 2.0 * Tstart
    if rhs == 0.0:
        return 0.0
    
    counter = 0
    myeq0 = root(rhs, T0, i)
    myeq1 = root(rhs, T1, i)
    
    while myeq0 > 0.0 and counter < 100:
        T0 /= 2.0
        counter += 1
        myeq0 = root(rhs, T0, i)
        if verbose:
            print(f"{T0} down {myeq0}")
    
    while myeq1 < 0.0 and counter < 100:
        T1 *= 2.0
        myeq1 = root(rhs, T1, i)
        counter += 1
        if verbose:
            print(f"{T1} up {myeq1}")
    
    if verbose:
        myeq0 = root(rhs, T0, i)
        myeq1 = root(rhs, T1, i)
        if myeq0 > myeq1:
            print("BUG in dychotomy")
    
    while abs(T1 - T0) > epsdycho and counter < 100:
        Taux = (T1 + T0) / 2.0
        counter += 1
        myeq0 = root(rhs, Taux, i)
        if myeq0 > 0.0:
            T1 = Taux
        else:
            T0 = Taux
        if verbose:
            myeq0 = root(rhs, T0, i)
            myeq1 = root(rhs, T1, i)
            print(f"{T0} middle {T1} {myeq0} {myeq1}")
    
    if counter > 99:
        print("Divergence in dichotomy")
    
    return (T1 + T0) / 2.0

# Generate temperature field
def genT():
    for i in range(Nz):
        rhs = rhsTeq(i)
        T[i] = getTbydycho(rhs, i, T[i])
        
        presfunc = 1.0
        inewton = 0
        while inewton < newton and abs(presfunc) > epsnewton:
            T0 = T[i]
            left = 0.0
            deriv = 0.0
            for j in range(1, jmax):
                dnu = nu[j] - nu[j-1]
                nu11 = (nu[j] + nu[j-1]) / 2.0
                kappaux = (1.0 - ascat(i * dz, nu[j])) * (kappanu[j] + kappanu[j-1]) / 2.0
                left += kappaux * BB(nu11, T0) * dnu
                deriv += kappaux * dBB(nu11, T0) * dnu
            
            presfunc = rhs - left
            if abs(deriv) > 1e-10:
                T[i] = T0 + presfunc / deriv
            
            if verbose:
                print(f"{i} T={T[i]} residue={presfunc} deriv={deriv} Tstefan={np.sqrt(np.sqrt(rhs * 15 * 2)) / 3.1416}")
        
        if inewton >= newton:
            print("Newton precision doubtful")


## ISIF algorithm

It is an initialization and then a loop of kmax iterations

At each iteration some information is printed. One should check the monotony

In [ ]:
# Multi-block iteration
def multiBlock(initT):
    for i in range(Nz):
        T[i] = initT
        for j in range(jmax):
            J0old[j][i] = 0.0
    
    for k in range(kmax):
        updateJ()
        genT()
        for i in range(Nz):
            for j in range(jmax):
                J0old[j][i] = J0[j][i]
        
        normG = 0.0
        for j in range(jmax):
            for i in range(Nz):
                normG += sqr(J0[j][i])*dz
        print(f"{k}     {T[2]*4798-273}     {normG/Bscale}")


## The main program

One can either do a single comutation with constant absorption equal to 0.5 or two comutations first with the Gemini absorption anf then with a modified absorption in some frequency range to simulate the presence of a GHG gas.

The "backtoz" function reinstall the physical altitude from the optical depth

In [ ]:
# Back to z function
def backtoz(z):
    return (np.sqrt(1.0 + 2.0 * drho * z) - 1.0) / drho

# Main function
def main():
    global jmax,J0,J0old
    jmax = readkappa(mykappafile)
    J0 =np.zeros((jmax,Nz))
    J0old =np.zeros((jmax,Nz))

    for K in range(2):
        for j in range(jmax):
            if K == 1:
                if nu[j] > 3.0 / 18.0 and nu[j] < 3.0 / 14.0:
                    kappanu[j] = 1.2 * kappanu[j]  # Visible R
            elif K == 2:
                kappanu[j] = 0.5  # constant kappa
        
        resstream = myresulttemperature + "5" + str(K) + ".txt"
        if lambda_val == 0:
            resstream = myresulttemperature + str(K) + ".txt"
        
        with open(resstream, 'w') as resultfile:
            print(f"\n iterations   T[2]    Norm of J0")
            t0 = time.time()
            multiBlock(Te / 2.0)
            elapsed = (time.time() - t0) / 60.0
            print(f" Time CPU = {elapsed:.6f}")
            
            print(f"\n tau [T]:")
            for i in range(0, Nz, 1):
                z = i * dz
                print(f"{backtoz(z):.4f} {T[i] * Tscale - 273:.4f}")
            
            for i in range(1, Nz):
                resultfile.write(f"{backtoz(i * dz):.4f} {T[i] * Tscale - 273:.4f}\n")
        
        rb = Cs * stefan * (Ts**4)
        for j in range(jmax):
            rb -= J0[j][Nz-1] * (nu[j] - nu[j-1])
        print(f" Radiation Budget {rb}")

# Helper function to get ascat(z, nu)
#def as_func(z, nu):
 #   return 0.3 + 0.3 * (z2 - z) * (z - z1) * (z > z1) * (z < z2) * 4.0 / (z1 - z2) + \
  #         0.3 * (nu < nu2) * (nu > nu1) * (4.0 * (nu - nu1) * (nu - nu2) / (nu2 - nu1))**2 * (z > z3)

if __name__ == "__main__":
    main()


Reading kappa(nu) from file _kappa.txt
Number of frequencies 552
Check Stefan id for the sun 12.877134011867136 13.8477782779757
Check Stefan id for the earth 8.681433368589914e-05 0.0000878697140573666

 iterations   T[2]    Norm of J0
0     -43.009624569781096     4.517301518359845e-06
1     -25.330878981343375     5.733830704380624e-06
2     -19.343172614566356     6.273404145650581e-06
3     -16.95640949145394     6.513996842020043e-06
4     -15.940784546713246     6.6203855845746724e-06
5     -15.497030433925943     6.667424522748877e-06
6     -15.301013212697512     6.688265892729244e-06
7     -15.214032345925204     6.6975175868874736e-06
8     -15.175361776476507     6.701629729524354e-06
9     -15.158155637670006     6.7034588439719036e-06
 Time CPU = 0.903283

 tau [T]:
-0.0000 -10.8001
0.0795 -12.3508
0.1641 -15.1582
0.2547 -18.4437
0.3529 -22.0795
0.4611 -26.2404
0.5829 -31.0549
0.7256 -36.2381
0.9059 -42.0427
1.2000 -48.1094
 Radiation Budget 0.0004492926373220585

 iterat